In [4]:
import torch
import operator
from typing import List
from transformers import LlamaForCausalLM, LlamaTokenizer

In [2]:
def hardware_backend(gm: torch.fx.GraphModule, example_inputs: List[torch.Tensor]):
    print("Generating hardware-specific code:")
    for node in gm.graph.nodes:
        print(f"Node: {node.op}, Target: {node.target}, Args: {node.args}")
        if node.op == 'call_function' and node.target == operator.add:
            node.target = torch.mul  # Replace the target function

    gm.graph.lint()
    gm.recompile()

    return lambda *inputs: gm(*inputs)

@torch.compile(backend=hardware_backend)
def fn(x, y):
    return x + y

x = torch.tensor(2)
y = torch.tensor(3)
result = fn(x, y)  # Should perform multiplication instead of addition
print(result)

Generating hardware-specific code:
Node: placeholder, Target: L_x_, Args: ()
Node: placeholder, Target: L_y_, Args: ()
Node: call_function, Target: <built-in function add>, Args: (l_x_, l_y_)
Node: output, Target: output, Args: ((add,),)
tensor(6)
